In [2]:
import sys
import cv2
import threading
import time
from Face_Detection import detect,upload
from PyQt5.QtWidgets import (
    QApplication, QLabel, QWidget, QVBoxLayout, QSizePolicy
)
from PyQt5.QtCore import Qt, QTimer
from PyQt5.QtGui import QImage, QPixmap

class RobotGUI(QWidget):
    def __init__(self):
        super().__init__()

        self.setWindowTitle("Robot Assistant")
        self.showFullScreen()
        self.setStyleSheet("background-color: black;")

        # Main display label
        self.display_label = QLabel(self)
        self.display_label.setAlignment(Qt.AlignCenter)
        self.display_label.setSizePolicy(QSizePolicy.Expanding, QSizePolicy.Expanding)
        self.display_label.setScaledContents(True)

        # Layout
        layout = QVBoxLayout()
        layout.setContentsMargins(0, 0, 0, 0)
        layout.setSpacing(0)
        layout.addWidget(self.display_label)
        self.setLayout(layout)

        # Webcam
        self.cap = cv2.VideoCapture(0)
        self.mode = "sleep"
        self.running = True

        # Timer to update UI
        self.timer = QTimer()
        self.timer.timeout.connect(self.update_gui)
        self.timer.start(30)

        # Simulated mode switching
        threading.Thread(target=self.simulate_modes, daemon=True).start()

    def update_gui(self):
        if self.mode in ["camera", "finger_counting"]:
            self.show_camera()
        elif self.mode == "sleep":
            self.show_text("😴", font_size=100, fg="white", bg="black")
        elif self.mode == "question":
            self.show_text("What is 2 + 2?", font_size=48)
        elif self.mode == "evaluating":
            self.show_text("Evaluating...", font_size=48)
        elif self.mode == "answer_prompt":
            self.show_text("Answer Now 🎤", font_size=48)

    def show_camera(self):
        ret, frame = self.cap.read()
        if ret:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            height, width, channels = frame.shape
            bytes_per_line = 3 * width
            q_img = QImage(frame.data, width, height, bytes_per_line, QImage.Format_RGB888)
            pixmap = QPixmap.fromImage(q_img).scaled(
                self.display_label.size(), Qt.IgnoreAspectRatio, Qt.SmoothTransformation
            )
            self.display_label.setPixmap(pixmap)

    def show_text(self, text, font_size=48, fg="black", bg="white"):
        self.display_label.setPixmap(QPixmap())  # clear any image
        self.display_label.setStyleSheet(
            f"color: {fg}; background-color: {bg}; font-size: {font_size}px;"
        )
        self.display_label.setText(text)

    def simulate_modes(self):
        while self.running:
            time.sleep(3); self.mode = "camera"
            time.sleep(3); self.mode = "answer_prompt"
            time.sleep(3); self.mode = "evaluating"
            time.sleep(3); self.mode = "finger_counting"
            time.sleep(3); self.mode = "sleep"

    def closeEvent(self, event):
        self.running = False
        if self.cap.isOpened():
            self.cap.release()
        event.accept()

if __name__ == "__main__":
    app = QApplication(sys.argv)
    gui = RobotGUI()
    gui.show()
    sys.exit(app.exec_())


ModuleNotFoundError: No module named 'Face_Detection'